# 03 · Error analysis

Compares the errors of TF-IDF (word+char) and RoBERTa (seed 42, no augmentation)
on the clean test conditions, then looks at attack failures, and exports a sample
of misclassified texts for **manual coding** (the structured manual error analysis).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir()) / "src"))
from report_utils import *

ROOT = project_root()
RES, DATA = ROOT / "results", ROOT / "data" / "processed"
FIG, TAB = output_dirs(ROOT)
_, LRS = load_results(RES)
pd.set_option("display.max_colwidth", 120)

KEYS = {"tfidf": "tfidf_word_char", "roberta": "roberta-base"}

def both_models(condition, aug=0.0):
    """One row per text with both detectors' scores and predictions."""
    out = None
    for short, key in KEYS.items():
        p = load_predictions(RES, DATA, key, condition, aug=aug,
                             lr=LRS.get(key))
        p = p.rename(columns={"score": f"score_{short}", "pred": f"pred_{short}"})
        out = p if out is None else out.merge(p[["id", f"score_{short}", f"pred_{short}"]], on="id")
    texts = pd.read_parquet(DATA / f"{condition}.parquet", columns=["id", "text"])
    out = out.merge(texts, on="id", how="left")
    out["condition"] = CONDITION_NAMES[condition]
    for short in KEYS:
        out[f"wrong_{short}"] = out[f"pred_{short}"] != out["label"]
    return out

clean = pd.concat([both_models(c) for c in ["test_seen", "test_unseen", "test_reppen"]],
                  ignore_index=True)
clean["class"] = np.where(clean["label"] == 1, "AI", "Human")
clean["words"] = word_count(clean["text"])
print(len(clean), "test predictions loaded")

## 1. Do the two detectors make the same mistakes?
For AI texts an error is a miss (false negative); for human texts it is a false accusation (false positive).

In [ ]:
def outcome(r):
    t, b = r["wrong_tfidf"], r["wrong_roberta"]
    return ("both wrong" if t and b else "only TF-IDF wrong" if t
            else "only RoBERTa wrong" if b else "both correct")

clean["outcome"] = clean.apply(outcome, axis=1)
agreement = (pd.crosstab([clean["condition"], clean["class"]], clean["outcome"])
             .reindex(columns=["both correct", "only TF-IDF wrong", "only RoBERTa wrong", "both wrong"],
                      fill_value=0))
save_table(agreement, "error_agreement", TAB, "Error overlap between TF-IDF (word+char) and RoBERTa, seed 42")
agreement

## 2. Error rates by domain and by text length

In [ ]:
def rates(df, by):
    g = df.groupby(by)
    return pd.DataFrame({
        "Texts": g.size(),
        "TF-IDF error %": g["wrong_tfidf"].mean() * 100,
        "RoBERTa error %": g["wrong_roberta"].mean() * 100,
    }).round(1)

by_domain = rates(clean, ["domain", "class"]).unstack("class")
by_domain.columns = [f"{m} ({c})" for m, c in by_domain.columns]
by_domain = by_domain[[c for c in by_domain.columns if not c.startswith("Texts")]]
save_table(by_domain, "errors_by_domain", TAB,
           "Error rate (%) by domain: misses on AI texts and false accusations on human texts")
by_domain

In [ ]:
clean["length"] = pd.cut(clean["words"], [0, 100, 200, 400, np.inf],
                         labels=["<100", "100-199", "200-399", "400+"], right=False)
by_length = rates(clean, ["length", "class"]).unstack("class")
by_length.columns = [f"{m} ({c})" for m, c in by_length.columns]
save_table(by_length, "errors_by_length", TAB, "Error rate (%) by text length in words")
by_length

## 3. Attack failures: examples
`repr()` shows invisible characters (zero-width spaces `\u200b`, extra line breaks).

In [ ]:
atk = pd.concat([both_models(c) for c in ["test_seen_attacks", "test_heldout_attacks"]],
                ignore_index=True)
atk_rates = atk.groupby(["attack", "label"])[["wrong_tfidf", "wrong_roberta"]].mean().unstack("label") * 100
atk_rates.columns = [f"{'TF-IDF' if m.endswith('tfidf') else 'RoBERTa'} "
                     f"{'missed AI' if l == 1 else 'flagged human'} %" for m, l in atk_rates.columns]
atk_rates.round(1)

In [ ]:
def show(df, n=2, chars=220):
    for _, r in df.head(n).iterrows():
        print(f"[{r['attack']} | {r['domain']} | label={'AI' if r['label'] else 'human'} "
              f"| TF-IDF={r['score_tfidf']:.3f} RoBERTa={r['score_roberta']:.3f}]")
        print(repr(r["text"][:chars]), "\n")

human = atk[atk["label"] == 0].sample(frac=1, random_state=0)
for attack, model in [("insert_paragraphs", "roberta"), ("zero_width_space", "roberta"),
                      ("paraphrase", "roberta"), ("homoglyph", "tfidf")]:
    print(f"##### Human texts wrongly flagged by {model.upper()} under '{attack}'")
    show(human[(human["attack"] == attack) & human[f"wrong_{model}"]])

## 4. Sample for manual coding
Exports up to 12 errors from each group to `results/error_sample_for_coding.csv`.
Open it in Excel or Numbers, read each text, and fill the **category** column with one
code from the list below (add a short note if useful). An existing file is never
overwritten, so your coding is safe.

| Code | Meaning |
|---|---|
| `formulaic_human` | Human text that is formal, list-like or template-based (recipes, abstracts, wiki) |
| `fluent_ai` | AI text that reads naturally, with no obvious repetition |
| `repetitive_human` | Human text with repetition or rhyme (e.g. poetry, song-like) |
| `short_text` | Too little text to judge (very short) |
| `noisy_formatting` | Odd formatting, broken text or leftover markup |
| `truncated` | Text cut off or ends mid-sentence |
| `topic_or_domain` | Unusual topic or domain-specific vocabulary |
| `label_doubt` | The label itself looks questionable |
| `other` | None of the above (explain in notes) |

In [ ]:
errors = clean[clean["outcome"] != "both correct"].copy()
errors["group"] = errors["outcome"] + " · " + errors["class"]
sample = pd.concat([g.sample(n=min(12, len(g)), random_state=0)
                    for _, g in errors.groupby("group")])
sample = sample[["group", "condition", "domain", "model", "class", "score_tfidf",
                 "score_roberta", "words", "text"]].round(4)
sample["text"] = sample["text"].str.slice(0, 2000)
sample["category"] = ""
sample["notes"] = ""

path = RES / "error_sample_for_coding.csv"
if path.exists():
    print(f"{path.name} already exists - not overwritten (delete it to export a new sample).")
else:
    sample.to_csv(path, index=False)
    print(f"Exported {len(sample)} errors to {path.name}")
print(sample["group"].value_counts().to_string())

## 5. Summary of the manual coding (run after filling the categories)

In [ ]:
coded = pd.read_csv(RES / "error_sample_for_coding.csv").fillna({"category": ""})
done = coded[coded["category"].str.strip() != ""]
print(f"{len(done)} of {len(coded)} errors coded")
if len(done):
    summary = pd.crosstab(done["category"], done["group"], margins=True, margins_name="Total")
    save_table(summary, "error_categories", TAB, "Manual error categories by error group")
    display(summary)